In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import random
import os.path as osp

from rdkit import Chem
import deepchem as dc
from deepchem.feat.smiles_tokenizer import BasicSmilesTokenizer
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
import torch_geometric.transforms as T
from torch_geometric.utils import negative_sampling, convert, to_dense_adj,structured_negative_sampling
from torch_geometric.nn import GCNConv, SAGEConv
from torch_geometric.nn.conv import MessagePassing

import torch
import torch.nn.functional as F
from torch.optim.lr_scheduler import ExponentialLR,MultiplicativeLR
from torch import Tensor
from torch.utils.data import DataLoader

from gensim.models import word2vec
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import pickle

from mol2vec.features import mol2alt_sentence, mol2sentence, MolSentence, DfVec
import torch
from sklearn.metrics import roc_auc_score ,auc,precision_recall_curve,f1_score

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
/home/giobbi/DDI_LLM_repo/DDI_with_ML/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
/home/giobbi/DDI_LLM_repo/DDI_with_ML/.venv/lib/python3.10/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing

# Interaction Data

In [2]:
DDI_graph = pd.read_csv('https://raw.githubusercontent.com/liiniix/BioSNAP/master/ChCh-Miner/ChCh-Miner_durgbank-chem-chem.tsv', sep='\t')
DDI_graph.rename(columns={'Drug1': 'src', 'Drug2': 'dst'}, inplace=True)
DrugIDs_in_graph = np.unique(DDI_graph.values) # there are 1514 unique drugs in the graph

In [3]:
G = nx.from_pandas_edgelist(DDI_graph, 'src', 'dst')

# Smiles

In [4]:
drugsSMILES = pd.read_csv('https://raw.githubusercontent.com/sshaghayeghs/molSMILES/main/structure%20links%202.csv')
drugID_smiles = drugsSMILES[["DrugBank ID", "SMILES"]]
drugID_smiles.dropna(inplace=True)
drugID_smiles.reset_index(drop=True, inplace=True)

/tmp/ipykernel_712027/706743108.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drugID_smiles.dropna(inplace=True)


# Description

In [5]:
drugsDESC = pd.read_csv('https://raw.githubusercontent.com/sshaghayeghs/molSMILES/main/Drug_description.csv')
drugID_DESC = drugsDESC[["Drug ID", "Discription"]]
drugID_DESC.dropna(inplace=True)
drugID_DESC.reset_index(drop=True, inplace=True)

/tmp/ipykernel_712027/1344816557.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drugID_DESC.dropna(inplace=True)


In [6]:
#checking if a molecule has a valid molecule corespodn to the smiles string
def is_valid_molecule(smiles) -> bool:
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None
    except:
        return False

In [7]:
valid_smiles = pd.DataFrame(drugID_smiles)
valid_smiles['IsValidMolecule'] = drugID_smiles['SMILES'].apply(is_valid_molecule)
df_valid_molecules = valid_smiles[valid_smiles['IsValidMolecule']]

# Drop the temporary 'IsValidMolecule' column
df_valid_molecules = df_valid_molecules.drop(columns=['IsValidMolecule'])

[12:35:49] Explicit valence for atom # 13 Cl, 5, is greater than permitted
[12:35:49] SMILES Parse Error: syntax error while parsing: OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(O)(O)O)C1=CC=C(C=C1)S([O-])([O-])[O-])\C1=CC=C(C=C1)S(O)(O)[O-]
[12:35:49] SMILES Parse Error: check for mistakes around position 84:
[12:35:49] C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(O
[12:35:49] ~~~~~~~~~~~~~~~~~~~~^
[12:35:49] SMILES Parse Error: extra open parentheses while parsing: OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(O)(O)O)C1=CC=C(C=C1)S([O-])([O-])[O-])\C1=CC=C(C=C1)S(O)(O)[O-]
[12:35:49] SMILES Parse Error: check for mistakes around position 40:
[12:35:49] 1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2
[12:35:49] ~~~~~~~~~~~~~~~~~~~~^
[12:35:49] SMILES Parse Error: extra open parentheses while parsing: OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C

In [8]:

allowed_drug=[list(df_valid_molecules['DrugBank ID']),list(drugID_DESC['Drug ID'])]
# There are 1278 drugIDs that occur in the graph. Some graph nodes do not have associated SMILES or drug description

#droping the links that do not have any SMILES
for l in allowed_drug:
  for index, row in DDI_graph.iterrows():
      # Check if both cells in the row are in the allowed cells list
      if row['src'] not in l or row['dst'] not in l:
          #b If either cell is not in the allowed cells list, remove the row
          DDI_graph.drop(index, inplace=True)



In [9]:
#27800 edges
DDI_graph=DDI_graph.reset_index(drop=True)


In [10]:
#save the drugs smiles and drug description in the networks into a new dataframe
drugID_smiles_ddi = drugID_smiles[drugID_smiles['DrugBank ID'].isin(list(np.unique(DDI_graph.values)))]
drugID_smiles_ddi=drugID_smiles_ddi.reset_index(drop=True)
drugID_DESC_ddi = drugID_DESC[drugID_DESC['Drug ID'].isin(list(np.unique(DDI_graph.values)))]
drugID_DESC_ddi=drugID_DESC_ddi.reset_index(drop=True)

In [12]:
def PyG_data(feature,DDI_graph):
  DrugIDs_in_graph = np.unique(DDI_graph.values)
  node_id_map = {node_name: i for i, node_name in enumerate(DrugIDs_in_graph)}
  # Replace node names with integer IDs in the edge list
  src = [node_id_map[node_name] for node_name in DDI_graph['src']]
  dst = [node_id_map[node_name] for node_name in DDI_graph['dst']]
  # Stack the arrays side by side to create a 2D array
  combined_array = np.column_stack((np.array(src), np.array(dst)))
  edge_index = []  # List of tuples representing edges between drugs
  for drug_1, drug_2 in combined_array:
    # Create an undirected graph by adding edges in both directions
    edge_index.append((drug_1, drug_2))
    edge_index.append((drug_2, drug_1))
  #Replace node names with integer IDs in the feature
  feature=torch.tensor(feature,dtype=torch.float32)
  data = Data(x=feature, edge_index=torch.tensor(edge_index).t().contiguous())
  return data

In [13]:
transform = RandomLinkSplit(num_val=0.2,
    num_test=0.2,
    is_undirected=True,
    add_negative_train_samples=False,
    neg_sampling_ratio=1.0)
#train_data, val_data, test_data = transform(data)

In [14]:
#### 5-2-5-Language models

In [15]:
allowed_drug=list(df_valid_molecules['DrugBank ID'])+list(drugID_DESC['Drug ID'])
def LM(DDI_graph,allowed_drug,model_name,dir,s):
    Drug=pd.read_csv(dir, sep=s,index_col=0)
    if 'Unnamed: 0' in Drug.columns:
      Drug.drop(columns='Unnamed: 0', inplace=True)
    df = Drug[Drug.iloc[:, 0].isin(allowed_drug)]
    df=df.reset_index(drop=True)
    if 'Discription' in df.columns:
      features=df.drop(df.columns[[0, 1, 2]], axis=1)
    else:
      features=df.drop(df.columns[[0, 1]], axis=1)
    print(model_name)
    return  features.values, DDI_graph

In [16]:
##6- GCN (Multi-view representation Fusion)


In [17]:
class Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, out_channels)
    def encode(self, x, edge_index):
        x=self.conv1(x, edge_index)
        x=F.dropout(x, p=0.3)
        x=F.relu(x)
        x=self.conv2(x, edge_index)
        x=F.dropout(x, p=0.3)
        x=F.relu(x)
        x=self.conv3(x, edge_index)
        return x

    def decode(self, z, edge_label_index):
        return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(
            dim=-1
        )  # product of a pair of nodes on each edge

    def decode_all(self, z):
        prob_adj = z @ z.t()
        return (prob_adj > 0).nonzero(as_tuple=False).t()
    
    def forward(self, x, edge_index, edge_label_index):
        z = self.encode(x, edge_index)
        return self.decode(z, edge_label_index)

In [ ]:
def train():
    model.train()
    optimizer.zero_grad()

    z = model.encode(train_data.x, train_data.edge_index) # initializing GCN model
    out = model.decode(z, edge_label_index).view(-1)
    loss = criterion(out, edge_label)
    loss.backward()
    optimizer.step()
    scheduler.step()
    return loss


@torch.no_grad()
def test(data):
    model.eval()
    z = model.encode(data.x, data.edge_index)
    out = model.decode(z, data.edge_label_index).view(-1).sigmoid()
    roc=roc_auc_score(data.edge_label.cpu().numpy(), out.cpu().numpy())
    label=data.edge_label.cpu().numpy()
    score=out.cpu().numpy()
    return roc,label,score

In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [20]:
def no_feature(smiles,DDI_graph):
  #DrugIDs_in_graph = np.unique(DDI_graph.values)
  features = np.ones((len(smiles),100))
  print('no_feature')
  return features,DDI_graph

In [ ]:
Embedding_models={
                #'No Feature':no_feature(drugID_smiles_ddi,DDI_graph),
                #'GPTSMILES':LM(DDI_graph,allowed_drug,'GPT+SMILES','/data/giobbi/embeddings/SMILES_GPT.csv','\t'),
                'GPTDesc':LM(DDI_graph,allowed_drug,'GPT+Desc','/data/giobbi/embeddings/Dr_Desc_GPT.csv','\t'),

                  #'Morgan':Morgan(drugID_smiles_ddi,DDI_graph),
                  #'Mol2vec':Mol2Vec(drugID_smiles_ddi,DDI_graph),
                  #'SPVec':character2vec(drugID_smiles_ddi,DDI_graph),
                  #'Doc2vec':doc2vec(drugID_smiles_ddi,DDI_graph),
                  
                  #'ChemBertaSMILES':LM(DDI_graph,allowed_drug,'Chemberta+SMILES','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank SMILES Embedding/SMILES_Chemberta.csv',','),
                  #'MolformerSMILES':LM(DDI_graph,allowed_drug,'Molformer+SMILES','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank SMILES Embedding/SMILES_Molformer.csv',','),

                  #'SBERTSMILES':LM(DDI_graph,allowed_drug,'SBERT+SMILES','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank SMILES Embedding/SBERT/SMILES_SBert.csv',','),
                  #'AngledBERTSMILES':LM(DDI_graph,allowed_drug,'AngledBERT+SMILES','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank SMILES Embedding/AngleBERT/SMILES_angleBert.csv',','),
                  #'LLaMASMILES':LM(DDI_graph,allowed_drug,'LLaMA+SMILES','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank SMILES Embedding/LLaMA/llama65b_base_SMILES_embeddings.csv','\t'),
                  #'LLaMA2SMILES':LM(DDI_graph,allowed_drug,'LLaMA2+SMILES','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank SMILES Embedding/LLaMA II/llamaII7b_base_SMILES_embeddings.csv','\t'),
                  #'AngledLLaMA2SMILES':LM(DDI_graph,allowed_drug,'AngledLLaMA+SMILES','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank SMILES Embedding/AngleLLaMA/SMILES_angleLlama.csv',','),
                  #'BERTDesc':LM(DDI_graph,allowed_drug,'BERT+Desc','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank Description Embedding/BERT/bert50mt_base_Discription_embeddings.csv','\t'),
                  #'SBERTDesc':LM(DDI_graph,allowed_drug,'SBERT+Desc','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank Description Embedding/SBERT/Desc_SBert.csv',','),
                  #'AngledBERTDesc':LM(DDI_graph,allowed_drug,'AngledBERT+Desc','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank Description Embedding/AngleBERT/Drug_description_angleBERT.csv',','),
                  #'LLaMADesc':LM(DDI_graph,allowed_drug,'LLaMA+Desc','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank Description Embedding/LLaMA/llama65b_base_Discription_embeddings.csv','\t'),
                  #'LLaMA2Desc':LM(DDI_graph,allowed_drug,'LLaMA2+Desc','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank Description Embedding/LLaMA II/llamaII7b_base_Discription_embeddings.csv','\t'),
                  #'AngledLLaMA2Desc':LM(DDI_graph,allowed_drug,'AngledLLaMA+Desc','/content/drive/MyDrive/Shaghayegh Sadeghi/Drug embedding/DrugBank Description Embedding/AngleLLaMA/Drug_description_angleLlama.csv',','),
                  }

GPT+Desc


In [29]:
lmbda = lambda epoch: 0.96

In [40]:
LR = [
    #0.01,
    #0.001,
    #0.0001,
    #0.0002,
    0.0003 #,0.00001
]

epochs = 100

patience = 10  # Number of epochs to wait for improvement
wait = 0
best_val_auc = final_test_auc = 0
best_model_state = None

modelname = [name for name in Embedding_models.keys()]
AUC = pd.DataFrame()
PR = pd.DataFrame()

AUC['Embedding'] = modelname
PR['Embedding'] = modelname

for l in LR:
    print('-------------------------------')
    print('=====Learning Rate:', l, '=======')
    print('-------------------------------')
    results_AUC = []
    results_PR = []
    for modelname, emb in Embedding_models.items():
        print('-------------------------------')
        print('=========', modelname, '=========')
        print('-------------------------------')
        data = PyG_data(emb[0], emb[1])
        train_data, val_data, test_data = transform(data)
        model = Net(data.num_features, 256, 256).to(device)
        optimizer = torch.optim.Adam(params=model.parameters(), lr=l)
        scheduler = MultiplicativeLR(optimizer, lr_lambda=lmbda)
        criterion = torch.nn.BCEWithLogitsLoss()
        '''
        neg_edge_index = negative_sampling(
            edge_index=train_data.edge_index, num_nodes=train_data.num_nodes,
            num_neg_samples=train_data.edge_label_index.size(1), method='sparse')
        '''
        struct_neg_tup = structured_negative_sampling(
            edge_index=train_data.edge_index,
            num_nodes=train_data.num_nodes,
            contains_neg_self_loops=False
        )
        neg_edge_index = torch.stack((struct_neg_tup[0], struct_neg_tup[2]), dim=0)
        neg_edge_index, _ = torch.unique(neg_edge_index, dim=1, return_inverse=True)

        edge_label_index = torch.cat(
            [train_data.edge_label_index, neg_edge_index],
            dim=-1,
        )
        edge_label = torch.cat([
            train_data.edge_label,
            train_data.edge_label.new_zeros(neg_edge_index.size(1))
        ], dim=0)
        best_val_auc = final_test_auc = 0
        for epoch in range(1, epochs):
            loss = train()
            val_auc = test(val_data)[0]
            test_auc = test(test_data)[0]
            label = test(test_data)[1]
            score = test(test_data)[2]
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                final_test_auc = test_auc
                best_scores = score
                wait = 0  # Reset wait counter
                best_model_state = model.state_dict()  # Save best model weights
            else:
                wait += 1
            print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val: {val_auc:.4f}')
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        # Load the best model state before evaluation
        if best_model_state is not None:
            model.load_state_dict(best_model_state)

        precision, recall, thresholds = precision_recall_curve(label, best_scores)
        pr = auc(recall, precision)
        results_AUC.append({"Embedding": emb, "AUC": final_test_auc})
        results_PR.append({"Embedding": emb, "PR_AUC": pr})
        #del data
        #del model

    # assign column values from the collected lists (use list comprehensions to extract values)
    AUC[str(l)] = [r["AUC"] for r in results_AUC]
    PR[str(l)] = [r["PR_AUC"] for r in results_PR]



desired_order = ['Embedding'] + [str(lr) for lr in LR]
AUC = AUC[desired_order]
PR = PR[desired_order]    

-------------------------------
=====Learning Rate: 0.0003 =======
-------------------------------
-------------------------------
========= GPTDesc =========
-------------------------------
Epoch: 001, Loss: 0.7039, Val: 0.5210
Epoch: 002, Loss: 0.6979, Val: 0.5764
Epoch: 003, Loss: 0.6952, Val: 0.6795
Epoch: 004, Loss: 0.6939, Val: 0.7658
Epoch: 005, Loss: 0.6933, Val: 0.8134
Epoch: 006, Loss: 0.6930, Val: 0.8487
Epoch: 007, Loss: 0.6927, Val: 0.8913
Epoch: 008, Loss: 0.6925, Val: 0.9079
Epoch: 009, Loss: 0.6923, Val: 0.9276
Epoch: 010, Loss: 0.6920, Val: 0.9414
Epoch: 011, Loss: 0.6916, Val: 0.9569
Epoch: 012, Loss: 0.6914, Val: 0.9620
Epoch: 013, Loss: 0.6910, Val: 0.9712
Epoch: 014, Loss: 0.6905, Val: 0.9750
Epoch: 015, Loss: 0.6900, Val: 0.9784
Epoch: 016, Loss: 0.6895, Val: 0.9791
Epoch: 017, Loss: 0.6890, Val: 0.9816
Epoch: 018, Loss: 0.6884, Val: 0.9805
Epoch: 019, Loss: 0.6876, Val: 0.9828
Epoch: 020, Loss: 0.6869, Val: 0.9833
Epoch: 021, Loss: 0.6862, Val: 0.9839
Epoch: 022,

In [71]:
dat = data
mod = model

In [87]:
drugID_DESC_ddi

,Drug ID,Discription
0,DB00006,Bivalirudin is a synthetic 20 residue peptide ...
1,DB00007,Leuprolide is a synthetic 9-residue peptide an...
2,DB00014,"Goserelin is a synthetic hormone. In men, it s..."
3,DB00035,"Desmopressin (dDAVP), a synthetic analogue of ..."
4,DB00080,Daptomycin is a cyclic lipopeptide antibacteri...
...,...,...
1318,DB11248,Zinc gluconate is a zinc salt of gluconic acid...
1319,DB11256,Levomefolic acid (INN) is the metabolite of fo...
1320,DB11315,Methscopolamine is a quaternary ammonium deriv...
1321,DB00873,;;;;


In [53]:
mod.eval()
with torch.no_grad():
    node_embeddings = mod.encode(d.x, d.edge_index)
node_embeddings = node_embeddings.cpu().numpy()

In [58]:
node_embeddings.shape

(8723, 256)

In [73]:
dat.x.shape

torch.Size([8723, 1536])

In [79]:
DrugIDs_in_graph = np.unique(DDI_graph.values)
node_id_map = {node_name: i for i, node_name in enumerate(DrugIDs_in_graph)}
id_node_map = {i: node_name for i, node_name in enumerate(DrugIDs_in_graph)}

In [83]:
id_node_map[1322]

'DB11315'

In [84]:
node_embeddings.shape

(8723, 256)

In [78]:
node_embeddings_df = pd.DataFrame(node_embeddings, index=[id_node_map[i] for i in range(node_embeddings.shape[0])])

KeyError: 1323

In [67]:
node_embeddings_df

,0,1,2,3,4,5,6,7,8,9,...,246,247,248,249,250,251,252,253,254,255
0,-0.024693,0.002213,-0.016680,-0.005881,0.006990,-0.014710,-0.016197,-0.009094,-0.032121,-0.010116,...,-0.008722,0.000574,-0.017576,0.005480,-0.013389,-0.009559,-0.011941,0.009208,0.007382,-0.002512
1,-0.057090,0.003687,-0.043154,-0.017650,0.009447,-0.034877,-0.045051,-0.019283,-0.069779,-0.021778,...,-0.021621,-0.005770,-0.040890,0.006584,-0.027062,-0.027143,-0.026128,0.017640,0.018881,-0.000902
2,-0.054396,0.004385,-0.041339,-0.016089,0.009114,-0.033623,-0.043548,-0.018334,-0.066343,-0.020279,...,-0.021021,-0.005266,-0.038926,0.006295,-0.026332,-0.026876,-0.024828,0.016122,0.018505,-0.000377
3,-0.042852,0.004176,-0.027117,-0.011367,0.008033,-0.022398,-0.030795,-0.011833,-0.050296,-0.016109,...,-0.012747,-0.001236,-0.030838,0.007226,-0.019764,-0.020618,-0.017390,0.011315,0.009243,-0.001671
4,-0.013244,0.001087,-0.011708,-0.001111,0.006850,-0.004635,-0.010722,-0.007693,-0.013758,-0.008788,...,-0.005850,0.000188,-0.009236,0.001833,-0.005780,-0.003836,-0.006110,0.001248,0.003792,-0.001287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8718,0.011963,0.026419,0.017642,0.010621,0.033508,0.030707,0.043994,-0.001952,-0.005469,-0.009298,...,-0.000969,-0.025857,-0.014271,0.003257,0.020984,0.053964,0.027983,-0.038970,-0.010498,-0.020467
8719,-0.004059,-0.017626,0.000084,-0.021195,0.016996,-0.007996,0.012162,0.018482,-0.008467,-0.005369,...,0.023255,-0.004200,-0.018447,0.021023,-0.011046,-0.001433,-0.007687,0.002369,-0.031604,0.009745
8720,-0.003763,-0.020307,-0.021772,-0.018961,-0.008113,0.007571,-0.032342,0.004068,-0.019617,-0.032707,...,0.018884,-0.014533,-0.006848,-0.042779,0.016719,0.041412,-0.008781,-0.020352,0.012294,-0.011951
8721,0.001099,0.008454,0.025446,-0.007072,-0.015285,0.020047,0.008507,0.011268,-0.021847,0.017790,...,0.012707,-0.022166,0.047820,0.042610,0.023279,-0.005670,-0.009578,-0.013002,0.006769,-0.011972


In [ ]:
from sklearn.manifold import TSNE

# Initialize t-SNE with 2 components for 2D visualization
tsne = TSNE(n_components=2 ) #, random_state=42) # 42

# Apply t-SNE to the drug embeddings
reduced_embeddings = tsne.fit_transform(node_embeddings)

# Convert the reduced embeddings to a DataFrame for easier handling
reduced_embeddings_df = pd.DataFrame(reduced_embeddings, columns=['TSNE-1', 'TSNE-2'])


In [62]:
import plotly.express as px

# reduced_embeddings_df should already exist from your t-SNE code
fig = px.scatter(
    reduced_embeddings_df,
    x='TSNE-1',
    y='TSNE-2',
    title='t-SNE Visualization of Drug Embeddings',
    width=800,
    height=600
)
fig.show()